# Lab 9 - Multimodal Fusion

Following this week's lecture, and combining what we have learnt in all previous weeks, we are now going to implement deep multimodal fusion.

Today, we are going to combine tabular data and image data to predict house prices.

We will use:
1.   Tabular property data which includes bedrooms, bathrooms, floor area, and postcode (zip code).
2.   House images which includes the front, bedroom, kitchen, and bathroom.
3.   TensorFlow to build custom branching neural network architectures.


Our tasks for today are as follows:
1.   A tabular-only model with a Dense Neural Network.
2.   An image model with a Convolutional Neural Network.
3.   An early fusion model.
4.   A late fusion model.


References
The data we are using comes from:
> Ahmed, E. and Moustafa, M., 2016. House price estimation from visual and textual features. arXiv preprint arXiv:1609.08399.

https://github.com/emanhamed/Houses-dataset


## Getting Started

**Before you begin:** navigate to the GitHub link above and download the dataset. Add all of the images to a folder called "images". Then, download "houses-tabular.csv" from this week's NOW Learning Room.




In [ ]:
!pip install -q tensorflow pandas numpy matplotlib scikit-learn pillow

print("Importing libraries...")
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import load_img, img_to_array
print("TensorFlow version:", tf.__version__)
print("Libraries imported successfully!")

# Loading the Data

I have formatted a .csv for this dataset for you (available in this week's NOW Learning Room). We will now load it and inspect it.

In [ ]:
tabular_path = "houses-tabular.csv"

houses_df = pd.read_csv(tabular_path)

print("Shape of tabular data:", houses_df.shape)
print("\nFirst five rows:")
display(houses_df.head())

In [ ]:
print("Column names:")
print(houses_df.columns.tolist())

print("\nData types:")
print(houses_df.dtypes)

print("Missing values per column:")
print(houses_df.isnull().sum())

# Collecting image filenames

Our data looks good. Now we will collect all of the frontal images of houses. In the naming conventions, the images are named "ID_room.jpg". E.g., the bathroom for house ID 1 is "1_bathroom.jpg".

We will include these image paths as a new feature, "Frontal Image Path".

In [ ]:
image_folder = "images"

houses_df["Frontal Image Path"] = houses_df["House ID"].astype(str) + "_frontal.jpg"
houses_df["Frontal Image Path"] = houses_df["Frontal Image Path"].apply(
    lambda x: os.path.join(image_folder, x)
)

display(houses_df.head())

Now let's inspect some random data from our dataset...

Note: a random seed is not given to sample, and therefore you can run this block of code multiple times to see different random samples.

**Point of reflection:** do some houses look more expensive than others based only on the image? For example, aesthetic design, architectural styles, size, driveways, gates, gardens?

This is why multimodal learning *might* be useful. Tabular information gives us structured facts, but the image might contain additional information that a formal spreadsheet cannot capture!

In [ ]:
sample_df = houses_df.sample(6)

plt.figure(figsize=(15, 10))

for i, (_, row) in enumerate(sample_df.iterrows()):
    img = load_img(row["Frontal Image Path"], target_size=(200, 200))

    plt.subplot(2, 3, i + 1)
    plt.imshow(img)
    plt.title(
        f'ID: {row["House ID"]}\n'
        f'Bedrooms: {row["Number of Bedrooms"]}, '
        f'Bathrooms: {row["Number of Bathrooms"]}\n'
        f'Area: {row["Area (m2)"]} m²\n'
        f'Price: {row["Price"]:,}'
    )
    plt.axis("off")

plt.tight_layout()
plt.show()

# Dataset Preparation

Now we are ready to prepare our data for learning. First, we will divide the features between tabular, images, and the target (price). Then, we are going to create train/test splits just as we have done previously.

In [ ]:
#Defining which features are tabular, images, and the output target
tabular_features = [
    "Number of Bedrooms",
    "Number of Bathrooms",
    "Area (m2)",
    "Zip Code"
]

X_tabular = houses_df[tabular_features].copy()
X_images = houses_df["Frontal Image Path"].copy()
y = houses_df["Price"].copy()

print("Tabular input shape:", X_tabular.shape)
print("Number of image paths:", len(X_images))
print("Target shape:", y.shape)



# Creating the training and testing splits
# Note that we have multiple "X_" inputs (one for each modality)
(
    X_tabular_train,
    X_tabular_test,
    X_images_train,
    X_images_test,
    y_train,
    y_test
) = train_test_split(
    X_tabular,
    X_images,
    y,
    test_size=0.2,
    random_state=42
)

print("Training tabular shape:", X_tabular_train.shape)
print("Testing tabular shape:", X_tabular_test.shape)
print("Training images:", len(X_images_train))
print("Testing images:", len(X_images_test))

Almost there... but before we begin, let's preprocess the data slightly. As we spoke about during the lecture, very large values can dominate learning. Therefore, we will alleviate this issue by standardising the tabular features.

In [ ]:
scaler_X = StandardScaler()

X_tabular_train_scaled = scaler_X.fit_transform(X_tabular_train)
X_tabular_test_scaled = scaler_X.transform(X_tabular_test)

print("Scaled training tabular shape:", X_tabular_train_scaled.shape)
print("Scaled testing tabular shape:", X_tabular_test_scaled.shape)

# Loading the images

Right now, our image feature is simply a directory and filename, i.e., a link to the image. Now we need to load the actual pixels from the image before applying computer vision to them.

**Note:** a standard input size is required for images. In this case, we will use 128px square images by resizing them.

**Point of reflection:** if you were to use a pre-trained CNN like VGG16, what image size would you use? (If you are unsure, refresh your memory by looking at the size of the input layer for VGG16).

In [ ]:
IMG_SIZE = (128, 128)

def load_and_preprocess_image(path):
    img = load_img(path, target_size=IMG_SIZE)
    img = img_to_array(img)
    img = img / 255.0
    return img

print("Loading training images...")
X_image_train = np.array([load_and_preprocess_image(path) for path in X_images_train])
print("Done.")

print("Loading testing images...")
X_image_test = np.array([load_and_preprocess_image(path) for path in X_images_test])
print("Done.")

print("Training image array shape:", X_image_train.shape)
print("Testing image array shape:", X_image_test.shape)

print("\nFull Summary:")
print("Tabular train:", X_tabular_train_scaled.shape)
print("Tabular test:", X_tabular_test_scaled.shape)
print("Image train:", X_image_train.shape)
print("Image test:", X_image_test.shape)
print("Target train:", y_train.shape)
print("Target test:", y_test.shape)

# Deep Learning

Our data is now ready for learning. We are going to build four models:
1.   Tabular only MLP
2.   Image only CNN
3.   Multimodal early fusion
4.   Multimodal late fusion

As this is a regression task (predicting a real number), we only need one output neuron with a linear activation. To score the model, we will use Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and the R^2 coefficient.

Below, I have also implemented a helper function for you which will calculate the results by using Scikit-learn's built-in metrics.



In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd

results = []

def evaluate_model(model_name, model, X_test, y_test):
    preds = model.predict(X_test, verbose=0).flatten()
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    results.append({
        "Model": model_name,
        "MAE": mae,
        "RMSE (£)": rmse,
        "R²": r2
    })

    print(f"{model_name} Results")
    print(f"MAE: £{mae:,.2f}")
    print(f"RMSE: £{rmse:,.2f}")
    print(f"R²: {r2:.4f}")
    return preds

# Tabular Model Training

We will now build a small MLP to learn from the tabular features. These are the number of bedrooms, bathrooms, area, and postal code. This will form our first baseline.

Note: pay close attention to the RMSE. This is measured in the same units as the expected output (e.g., dollars or pounds).

In [ ]:
def tabular_mlp(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(32, activation="relu"),
        layers.Dense(16, activation="relu"),
        layers.Dense(8, activation="relu"),
        layers.Dense(1)
    ])

    model.compile(
        optimizer="adam",
        loss="mse",
        metrics=["mae"]
    )

    return model

tabular_model = tabular_mlp(X_tabular_train_scaled.shape[1])
tabular_model.summary()


history_tabular = tabular_model.fit(
    X_tabular_train_scaled,
    y_train,
    validation_data=(X_tabular_test_scaled, y_test),
    epochs=50,
    batch_size=16,
    verbose=2
)


tabular_preds = evaluate_model(
    "Tabular Data MLP",
    tabular_model,
    X_tabular_test_scaled,
    y_test
)

# CNN Training

Now we will build a small CNN that learns from only the frontal house images. As part of our ablation studies, this will give us the second part that will later be fused together.

**Note:** unlike several weeks ago, we are now using the Keras Functional API. This allows for greater flexibility in the shape of neural networks, allowing us to define how layers are connected together. Format:

> layer_name = xyz(connected_to_layer)

Read about that here: https://keras.io/guides/functional_api/

In [ ]:
def image_cnn(input_shape):
    image_input = layers.Input(shape=input_shape)

    x = layers.Conv2D(16, (3, 3), activation="relu", padding="same")(image_input)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, (3, 3), activation="relu", padding="same")(x)
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(16, activation="relu")(x)
    output = layers.Dense(1)(x)

    model = keras.Model(inputs=image_input, outputs=output)

    model.compile(
        optimizer="adam",
        loss="mse",
        metrics=["mae"]
    )

    return model

image_model = image_cnn(X_image_train.shape[1:])
image_model.summary()

history_image = image_model.fit(
    X_image_train,
    y_train,
    validation_data=(X_image_test, y_test),
    epochs=50,
    batch_size=16,
    verbose=2
)

image_preds = evaluate_model(
    "Image CNN",
    image_model,
    X_image_test,
    y_test
)

# Early Fusion

So far, we have trained two neural networks - an MLP that learns from tabular data and a CNN that learns from images. Now we are going to build an early fusion network that takes both tabular and image data as input.

Our models will remain the same, but they will now train as a branched model.

In [ ]:
def early_fusion(tabular_dim, image_shape):
    # Tabular branch input
    tabular_input = layers.Input(shape=(tabular_dim,))
    t = layers.Dense(32, activation="relu")(tabular_input)
    t = layers.Dense(16, activation="relu")(t)
    t = layers.Dense(8, activation="relu")(t)

    # Image branch input
    image_input = layers.Input(shape=image_shape)
    i = layers.Conv2D(16, (3, 3), activation="relu", padding="same")(image_input)
    i = layers.MaxPooling2D()(i)
    i = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(i)
    i = layers.MaxPooling2D()(i)
    i = layers.Conv2D(64, (3, 3), activation="relu", padding="same")(i)
    i = layers.GlobalAveragePooling2D()(i)
    i = layers.Dense(16, activation="relu")(i)

    # Early fusion (combining the branches into one)
    combined = layers.concatenate([t, i])
    x = layers.Dense(16, activation="relu")(combined)
    x = layers.Dense(8, activation="relu")(x)
    output = layers.Dense(1)(x)

    model = keras.Model(inputs=[tabular_input, image_input], outputs=output)

    model.compile(
        optimizer="adam",
        loss="mse",
        metrics=["mae"]
    )

    return model

early_fusion_model = early_fusion(
    X_tabular_train_scaled.shape[1],
    X_image_train.shape[1:]
)

early_fusion_model.summary()

This model is a little more complicated, and so it is best practice to generate a diagram so we can see what it actually looks like...

**Inspect the below diagram before moving on!**

In [ ]:
!pip install -q pydot

from tensorflow.keras.utils import plot_model

plot_model(
    early_fusion_model,
    to_file="early_fusion_model.png",
    show_shapes=True,
    show_layer_names=True,
    expand_nested=True,
    dpi=100
)

from IPython.display import Image
Image("early_fusion_model.png")

# Training the early fusion model

Now that we have built our early fusion model, let's train it on the data and see how it performs!

In [ ]:
history_early = early_fusion_model.fit(
    [X_tabular_train_scaled, X_image_train],
    y_train,
    validation_data=([X_tabular_test_scaled, X_image_test], y_test),
    epochs=50,
    batch_size=16,
    verbose=1
)

early_preds = evaluate_model(
    "Early Fusion",
    early_fusion_model,
    [X_tabular_test_scaled, X_image_test],
    y_test
)

# Late Fusion

Last but not least, we will now implement a very simple method for late fusion. As we discussed in the lecture, we will use averaging. Simply, this is the prediction of all models divided by the total number of models.

**Further reading:** after implementing late fusion, read about more complex types of ensembles here - https://scikit-learn.org/stable/api/sklearn.ensemble.html

**Point of reflection:** how might voting increase the ability of late fusion beyond averaging? Think back to **weighting** from last week's lecture...

In [ ]:
late_preds = (tabular_preds + image_preds) / 2
late_mae = mean_absolute_error(y_test, late_preds)
late_rmse = np.sqrt(mean_squared_error(y_test, late_preds))
late_r2 = r2_score(y_test, late_preds)

results.append({
    "Model": "Late Fusion (Average)",
    "MAE (£)": late_mae,
    "RMSE (£)": late_rmse,
    "R²": late_r2
})

print("Late Fusion (Average) Results")
print(f"MAE: £{late_mae:,.2f}")
print(f"RMSE: £{late_rmse:,.2f}")
print(f"R²: {late_r2:.4f}")

# Results Summary

After each model is trained, we have been storing our results in the results list. The below code will sort each by the RMSE (lower is better).

**Point of reflection:** which model performed best? Which was worst?
**Something to think about:** was fusion worth it?

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="RMSE (£)")
results_df

# Your Task

Now it's your turn. Your task is to add a new set of images to the model. You can choose any other room to add (e.g., bathroom, bedroom, or kitchen).

To implement this, you will need to:
1.   Train a new CNN on the images only (e.g., a kitchen CNN).
2.   Add the new CNN as a branch to the early fusion model (now with three inputs instead of two!)
3.   Explore late fusion of the new three modalities. For example:
```
late_preds = (tabular_preds + image_preds + kitchen_preds) / 3
```


---

# Stretch Challenge
Once you are confident adding a new branch to your network, try to build an early fusion approach that learns from **ALL** of the available data.

For example:

Tabular > MLP_1

Frontal > CNN_1

Kitchen > CNN_2

Bathroom > CNN_3

Bedroom > CNN_4

Concatenate(MLP_1, CNN_1, CNN_2, CNN_3, CNN_4) > MLP > Prediction



---

# Coursework
You may also wish to use this time to work on your coursework.

